In [1]:
import os
import numpy as np
import mne

from mne.decoding import CSP

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, f1_score

In [2]:
data_folder = "../data/BCI_IV_2a"

subjects = [
    "A01T",
    "A02T",
    "A03T",
    "A04T",
    "A05T",
    "A06T",
    "A07T",
    "A08T",
    "A09T"
]

# Fixed OUTPUT labels used throughout this notebook.
# IMPORTANT: MNE assigns per-file numeric codes to annotation
# descriptions based on which descriptions appear in that specific
# file and in what order -- the same description (e.g. "769") can get
# a DIFFERENT numeric code in different subjects' files (confirmed for
# A04T in A04T_Investigation.ipynb). We always look codes up by
# description string per file (see loading cell below) and translate
# them to this fixed set of output labels, so labels are consistent
# across every subject.
DESC_TO_LABEL = {
    "769": 7,   # LEFT
    "770": 8,   # RIGHT
    "771": 9,   # FOOT
    "772": 10,  # TONGUE
}


In [3]:
X_list = []
y_list = []
groups_list = []

for subject in subjects:

    print("==============================")
    print("Processing:", subject)
    print("==============================")

    file_path = os.path.join(
        data_folder,
        subject + ".gdf"
    )

    raw = mne.io.read_raw_gdf(
        file_path,
        preload=True,
        verbose=False
    )

    # Keep EEG channels only
    raw.pick_types(eeg=True)

    # Find events. event_dict maps annotation DESCRIPTION -> per-file
    # numeric code. We must look codes up by description, not assume
    # a fixed value (see DESC_TO_LABEL cell above).
    events, event_dict = mne.events_from_annotations(
        raw,
        verbose=False
    )

    print("Events:", event_dict)

    # Build this subject's event_id using description strings
    available_events = {
        desc: event_dict[desc]
        for desc in DESC_TO_LABEL
        if desc in event_dict
    }

    if len(available_events) != 4:
        missing = set(DESC_TO_LABEL) - set(available_events)
        print(f"WARNING: {subject} missing motor-imagery events: {missing}")

    print("Using:", available_events)

    # Epoch: 1-3 seconds
    epochs = mne.Epochs(
        raw,
        events,
        event_id=available_events,
        tmin=1,
        tmax=3,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()

    # Remap this subject's per-file codes to the fixed output labels
    code_to_label = {
        event_dict[desc]: DESC_TO_LABEL[desc]
        for desc in available_events
    }
    y = np.array([code_to_label[code] for code in epochs.events[:, -1]])

    print("X:", X.shape)
    print("y:", y.shape)
    print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

    # Keep only the first 22 EEG channels
    X = X[:, :22, :]

    # 8-30 Hz filtering
    X_filtered_subject = np.empty_like(X)

    for i in range(X.shape[0]):

        X_filtered_subject[i] = mne.filter.filter_data(
            X[i],
            sfreq=250,
            l_freq=8,
            h_freq=30,
            verbose=False
        )

    X_list.append(X_filtered_subject)
    y_list.append(y)

    # Store subject ID for every trial
    groups_list.extend(
        [subject] * len(y)
    )


# Combine all subjects
X_filtered = np.concatenate(
    X_list,
    axis=0
)

y_multi = np.concatenate(
    y_list,
    axis=0
)

groups = np.array(
    groups_list
)

print("\n================================")
print("FINAL DATASET")
print("================================")

print("X_filtered:", X_filtered.shape)
print("y_multi:", y_multi.shape)
print("groups:", groups.shape)

print(
    "Classes:",
    np.unique(
        y_multi,
        return_counts=True
    )
)

print(
    "Subjects:",
    np.unique(
        groups,
        return_counts=True
    )
)


Processing: A01T


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Events: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using: {'769': 7, '770': 8, '771': 9, '772': 10}
X: (288, 25, 501)
y: (288,)
Class counts: {np.int64(7): np.int64(72), np.int64(8): np.int64(72), np.int64(9): np.int64(72), np.int64(10): np.int64(72)}
Processing: A02T


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Events: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using: {'769': 7, '770': 8, '771': 9, '772': 10}
X: (288, 25, 501)
y: (288,)
Class counts: {np.int64(7): np.int64(72), np.int64(8): np.int64(72), np.int64(9): np.int64(72), np.int64(10): np.int64(72)}
Processing: A03T


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Events: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using: {'769': 7, '770': 8, '771': 9, '772': 10}
X: (288, 25, 501)
y: (288,)
Class counts: {np.int64(7): np.int64(72), np.int64(8): np.int64(72), np.int64(9): np.int64(72), np.int64(10): np.int64(72)}
Processing: A08T


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Events: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using: {'769': 7, '770': 8, '771': 9, '772': 10}
X: (288, 25, 501)
y: (288,)
Class counts: {np.int64(7): np.int64(72), np.int64(8): np.int64(72), np.int64(9): np.int64(72), np.int64(10): np.int64(72)}
Processing: A09T


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Events: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using: {'769': 7, '770': 8, '771': 9, '772': 10}
X: (288, 25, 501)
y: (288,)
Class counts: {np.int64(7): np.int64(72), np.int64(8): np.int64(72), np.int64(9): np.int64(72), np.int64(10): np.int64(72)}

FINAL DATASET
X_filtered: (2592, 22, 501)
y_multi: (2592,)
groups: (2592,)
Classes: (array([ 7,  8,  9, 10]), array([648, 648, 648, 648]))
Subjects: (array(['A01T', 'A02T', 'A03T', 'A04T', 'A05T', 'A06T', 'A07T', 'A08T',
       'A09T'], dtype='<U4'), array([288, 288, 288, 288, 288, 288, 288, 288, 288]))


In [4]:
print("X_filtered:", X_filtered.shape)
print("y_multi:", y_multi.shape)
print("groups:", groups.shape)

print("\nClass distribution:")
print(np.unique(y_multi, return_counts=True))

print("\nSubject distribution:")
print(np.unique(groups, return_counts=True))

X_filtered: (2592, 22, 501)
y_multi: (2592,)
groups: (2592,)

Class distribution:
(array([ 7,  8,  9, 10]), array([648, 648, 648, 648]))

Subject distribution:
(array(['A01T', 'A02T', 'A03T', 'A04T', 'A05T', 'A06T', 'A07T', 'A08T',
       'A09T'], dtype='<U4'), array([288, 288, 288, 288, 288, 288, 288, 288, 288]))


In [5]:
logo = LeaveOneGroupOut()

fold_results = []

for train_idx, test_idx in logo.split(
    X_filtered,
    y_multi,
    groups=groups
):

    held_out_subject = groups[test_idx[0]]

    X_train = X_filtered[train_idx]
    X_test = X_filtered[test_idx]

    y_train = y_multi[train_idx]
    y_test = y_multi[test_idx]

    print("==============================")
    print("TEST SUBJECT:", held_out_subject)
    print("==============================")

    # -------------------------
    # CSP
    # -------------------------

    csp = CSP(
        n_components=6,
        reg=None,
        log=True,
        norm_trace=False
    )

    X_train_csp = csp.fit_transform(
        X_train,
        y_train
    )

    X_test_csp = csp.transform(
        X_test
    )

    # -------------------------
    # NORMALIZATION
    # -------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_csp
    )

    X_test_scaled = scaler.transform(
        X_test_csp
    )

    # -------------------------
    # LDA
    # -------------------------

    lda = LinearDiscriminantAnalysis()

    lda.fit(
        X_train_scaled,
        y_train
    )

    y_pred = lda.predict(
        X_test_scaled
    )

    # -------------------------
    # RESULTS
    # -------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    fold_results.append(
        (
            held_out_subject,
            accuracy,
            macro_f1
        )
    )

    print(
        f"Accuracy: {accuracy * 100:.2f}%"
    )

    print(
        f"Macro F1: {macro_f1:.4f}"
    )


# ==============================
# FINAL RESULTS
# ==============================

accuracies = [
    x[1] for x in fold_results
]

f1_scores = [
    x[2] for x in fold_results
]

print("\n================================")
print("FINAL NORMALIZATION RESULTS")
print("================================")

for subject, acc, f1 in fold_results:

    print(
        f"{subject}: "
        f"{acc * 100:.2f}% | "
        f"F1: {f1:.4f}"
    )

print(
    f"\nMean Accuracy: "
    f"{np.mean(accuracies) * 100:.2f}%"
)

print(
    f"Std Accuracy: "
    f"{np.std(accuracies) * 100:.2f}%"
)

print(
    f"Mean Macro F1: "
    f"{np.mean(f1_scores):.4f}"
)

TEST SUBJECT: A01T
Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=7 covariance using EMPIRICAL
Done.
Estimating class=8 covariance using EMPIRICAL
Done.
Estimating class=9 covariance using EMPIRICAL
Done.
Estimating class=10 covariance using EMPIRICAL
Done.
Accuracy: 48.61%
Macro F1: 0.4178
TEST SUBJECT: A02T
Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=7 covariance using EMPIRICAL
Done.
Estimating class=8 covariance using EMPIRICAL
Done.
Estimating class=9 covariance using EMPIRICAL
Done.
Estimating class=10 covariance using EMPIRICAL
Done.
Accuracy